In [37]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from tqdm import tqdm
from DATA.stock_invest_function import *
from statsmodels.tsa.statespace.sarimax import SARIMAX
from datetime import datetime
from itertools import product
from sqlalchemy import create_engine, text
import sqlalchemy
import matplotlib
import matplotlib.pyplot as plt
import warnings

# ===================== 기본 설정 =====================
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

LOG_TRANSFORM_CODES = ['851762']   # 로그 변환 적용 HS Code
FORECAST_STEPS = 15                # 예측 개월 수
MIN_PERIODS = 60                   # 최소 데이터 개수(5년)

# DB 접속 정보
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# 실행 날짜
input_date = datetime.now().date()

print(f"=== SARIMA 예측 시스템 시작 ({input_date}) ===")
print(f"로그 변환 적용 HS Code: {LOG_TRANSFORM_CODES}")

# SQLAlchemy 엔진
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)


# ======================================================================
# === [PATCH 1] : 중복 방지/스키마 보강 유틸 함수 (분기/월별 공통) - id 없이 동작
# ======================================================================
def _dedup_monthly(conn):
    """
    (hs_code_6d, date, forecast_flag) 기준 중복 제거
    - forecast_flag NULL -> 0
    - (input_date, created_at) 최신만 보존
    """
    # created_at 없을 수 있으니 선보강
    conn.execute(text("""
        ALTER TABLE us_trade_monthly_data_with_forecast
        ADD COLUMN IF NOT EXISTS created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    """))
    # NULL 통일
    conn.execute(text("""
        UPDATE us_trade_monthly_data_with_forecast
        SET forecast_flag = 0
        WHERE forecast_flag IS NULL
    """))
    # JOIN 기반 삭제: 더 '오래된' 행 삭제
    conn.execute(text("""
        DELETE t
        FROM us_trade_monthly_data_with_forecast AS t
        JOIN us_trade_monthly_data_with_forecast AS t2
          ON t.hs_code_6d = t2.hs_code_6d
         AND t.`date`     = t2.`date`
         AND IFNULL(t.forecast_flag,0) = IFNULL(t2.forecast_flag,0)
         AND (
              t.input_date < t2.input_date
           OR (t.input_date = t2.input_date AND t.created_at < t2.created_at)
         )
    """))

def _dedup_quarterly(conn):
    """
    (hs_code_6d, quarter, forecast_flag) 기준 중복 제거
    - forecast_flag NULL -> 0
    - (input_date, created_at) 최신만 보존
    """
    # created_at 보강
    conn.execute(text("""
        ALTER TABLE us_trade_quarter_data_with_forecast
        ADD COLUMN IF NOT EXISTS created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    """))
    # NULL 통일
    conn.execute(text("""
        UPDATE us_trade_quarter_data_with_forecast
        SET forecast_flag = 0
        WHERE forecast_flag IS NULL
    """))
    # JOIN 기반 삭제: 더 '오래된' 행 삭제
    conn.execute(text("""
        DELETE t
        FROM us_trade_quarter_data_with_forecast AS t
        JOIN us_trade_quarter_data_with_forecast AS t2
          ON t.hs_code_6d = t2.hs_code_6d
         AND t.quarter    = t2.quarter
         AND IFNULL(t.forecast_flag,0) = IFNULL(t2.forecast_flag,0)
         AND (
              t.input_date < t2.input_date
           OR (t.input_date = t2.input_date AND t.created_at < t2.created_at)
         )
    """))


def ensure_unique_keys_monthly(conn):
    """월별 테이블 비즈니스 유니크 보장 + 필요시 자동 정리/재시도 (id 미사용)"""
    # 보정: created_at 없으면 먼저 추가 (UNIQUE 전 정리에 필요)
    conn.execute(text("""
        ALTER TABLE us_trade_monthly_data_with_forecast
        ADD COLUMN IF NOT EXISTS created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    """))
    conn.execute(text("""
        ALTER TABLE us_trade_monthly_data_with_forecast
        MODIFY forecast_flag TINYINT(1) NOT NULL DEFAULT 0
    """))

    try:
        conn.execute(text("""
            ALTER TABLE us_trade_monthly_data_with_forecast
            ADD UNIQUE KEY uniq_monthly_biz (hs_code_6d, `date`, forecast_flag)
        """))
        print("월별: uniq_monthly_biz UNIQUE KEY 추가 완료")
    except Exception as e:
        msg = str(e).lower()
        if "duplicate" in msg or "exists" in msg:
            print("월별: uniq_monthly_biz 추가 실패(중복/기존 존재). 중복 정리 후 재시도...")
            _dedup_monthly(conn)
            try:
                conn.execute(text("""
                    ALTER TABLE us_trade_monthly_data_with_forecast
                    ADD UNIQUE KEY uniq_monthly_biz (hs_code_6d, `date`, forecast_flag)
                """))
                print("월별: uniq_monthly_biz UNIQUE KEY 재시도 성공")
            except Exception as e2:
                print(f"월별: uniq_monthly_biz UNIQUE KEY 재시도 실패: {e2}")
        else:
            raise

def ensure_unique_keys_quarterly(conn):
    """분기 테이블 비즈니스 유니크 보장 + 필요시 자동 정리/재시도 (id 미사용)"""
    # 보정: created_at 없으면 먼저 추가 (UNIQUE 전 정리에 필요)
    conn.execute(text("""
        ALTER TABLE us_trade_quarter_data_with_forecast
        ADD COLUMN IF NOT EXISTS created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    """))
    conn.execute(text("""
        ALTER TABLE us_trade_quarter_data_with_forecast
        MODIFY forecast_flag TINYINT(1) NOT NULL DEFAULT 0
    """))

    try:
        conn.execute(text("""
            ALTER TABLE us_trade_quarter_data_with_forecast
            ADD UNIQUE KEY uniq_quarter_biz (hs_code_6d, quarter, forecast_flag)
        """))
        print("분기: uniq_quarter_biz UNIQUE KEY 추가 완료")
    except Exception as e:
        msg = str(e).lower()
        if "duplicate" in msg or "exists" in msg:
            print("분기: uniq_quarter_biz 추가 실패(중복/기존 존재). 중복 정리 후 재시도...")
            _dedup_quarterly(conn)
            try:
                conn.execute(text("""
                    ALTER TABLE us_trade_quarter_data_with_forecast
                    ADD UNIQUE KEY uniq_quarter_biz (hs_code_6d, quarter, forecast_flag)
                """))
                print("분기: uniq_quarter_biz UNIQUE KEY 재시도 성공")
            except Exception as e2:
                print(f"분기: uniq_quarter_biz UNIQUE KEY 재시도 실패: {e2}")
        else:
            raise
# ======================================================================

# ===================== 분기 테이블 존재/스키마 보강 =====================
print("데이터베이스 테이블 확인 중...")

with engine.connect() as conn:
    # us_trade_quarter_data_with_forecast 테이블 존재 여부
    table_exists_query = """
    SELECT COUNT(*) as count
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
    AND table_name = 'us_trade_quarter_data_with_forecast'
    """
    result = conn.execute(text(table_exists_query))
    table_exists = result.fetchone()[0] > 0

    if not table_exists:
        create_table_query = """
        CREATE TABLE us_trade_quarter_data_with_forecast (
            id INT AUTO_INCREMENT PRIMARY KEY,
            hs_code_6d VARCHAR(10) NOT NULL,
            quarter VARCHAR(10) NOT NULL,
            expDlr FLOAT NOT NULL,
            date DATE NOT NULL,
            input_date DATE NOT NULL,
            forecast_flag TINYINT(1) NOT NULL DEFAULT 0,        -- [PATCH: NOT NULL]
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE KEY unique_record (hs_code_6d, quarter, input_date),
            INDEX idx_hs_code_date (hs_code_6d, date),
            INDEX idx_input_date (input_date),
            UNIQUE KEY uniq_quarter_biz (hs_code_6d, quarter, forecast_flag)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        conn.execute(text(create_table_query))
        print("새 분기 테이블 생성 완료")
    else:
        # 컬럼 리스트
        column_check_query = """
        SELECT COLUMN_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = 'us_trade_quarter_data_with_forecast'
        """
        result = conn.execute(text(column_check_query))
        existing_columns = [row[0] for row in result.fetchall()]

        if 'input_date' not in existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_quarter_data_with_forecast "
                "ADD COLUMN input_date DATE NOT NULL DEFAULT '2025-01-01'"
            ))
            print("분기 테이블에 input_date 컬럼 추가 완료")

        if 'forecast_flag' not in existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_quarter_data_with_forecast "
                "ADD COLUMN forecast_flag TINYINT(1) NOT NULL DEFAULT 0"  # [PATCH: NOT NULL]
            ))
            print("분기 테이블에 forecast_flag 컬럼 추가 완료")

        if 'created_at' not in existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_quarter_data_with_forecast "
                "ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
            ))
            print("분기 테이블에 created_at 컬럼 추가 완료")

        # (기존 코드의 try/except 대신 견고한 유틸 호출)
        ensure_unique_keys_quarterly(conn)  # === [PATCH 2] 분기 유니크 보강/정리

    conn.commit()

print("분기 테이블 설정 완료")

# ===================== 무역 데이터 로딩 =====================
print("무역 데이터 로딩 중...")
trade_df = fetch_table_data(db_info, "us_trade_data")
trade_df['hs_code_6d'] = trade_df['hs_code'].astype(str).str[:6]
trade_df['date'] = pd.to_datetime(trade_df['date'])

print(f"총 {len(trade_df):,}개 레코드 로드 완료")
print(f"유효한 HS Code (6자리): {len(trade_df['hs_code_6d'].unique()):,}개")

valid_codes = trade_df['hs_code_6d'].unique().tolist()

# ===================== SARIMA 함수 =====================
def forecast_sarima(df, date_col='date', value_col='expDlr', steps=14, use_log=False):
    try:
        ts = df.groupby(date_col)[value_col].sum().asfreq('M')
        if ts.isnull().any() or len(ts.dropna()) < MIN_PERIODS:
            return pd.Series(dtype='float64')

        if use_log:
            ts = np.log(ts)

        p = d = q = [0, 1]
        P = D = Q = [0, 1]
        s = 12

        param_combinations = list(product(p, d, q))
        seasonal_combinations = list(product(P, D, Q))
        total_combinations = list(product(param_combinations, seasonal_combinations))

        best_aic = np.inf
        best_model = None

        for (order, seasonal) in total_combinations:
            seasonal_order = (*seasonal, s)
            try:
                model = SARIMAX(ts, order=order, seasonal_order=seasonal_order)
                result = model.fit(disp=False, maxiter=50)
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_model = result
            except Exception:
                continue

        if best_model is None:
            return pd.Series(dtype='float64')

        forecast = best_model.forecast(steps=steps)
        if use_log:
            forecast = np.exp(forecast)

        forecast.index = pd.date_range(
            start=ts.index[-1] + pd.offsets.MonthEnd(1),
            periods=steps,
            freq='M'
        )
        return forecast

    except Exception:
        return pd.Series(dtype='float64')

# ===================== SARIMA 예측 실행 =====================
print(f"총 {len(valid_codes):,}개 HS Code에 대해 SARIMA 예측 시작...")

forecast_list = []
model_count = 0
log_used_count = 0

for code in tqdm(valid_codes, desc="SARIMA 예측 진행"):
    sub_df = trade_df[trade_df['hs_code_6d'] == code].copy()
    use_log = code in LOG_TRANSFORM_CODES
    if use_log:
        log_used_count += 1

    forecast = forecast_sarima(
        sub_df[['date', 'expDlr']],
        steps=FORECAST_STEPS,
        use_log=use_log
    )

    if not forecast.empty:
        temp_df = pd.DataFrame({
            'hs_code_6d': code,
            'date': forecast.index,
            'expDlr': forecast.values,
            'forecast_flag': 1,
            'input_date': input_date
        })
        forecast_list.append(temp_df)
        model_count += 1

print(f"예측 완료: {model_count}개 HS Code")
print(f"로그 변환 사용: {log_used_count}개 HS Code")

# ===================== 과거/예측 월 데이터 결합 =====================
historical_df = trade_df.groupby(['hs_code_6d', 'date'], as_index=False)['expDlr'].sum()
historical_df['forecast_flag'] = 0
historical_df['input_date'] = input_date

if forecast_list:
    forecast_combined = pd.concat(forecast_list, ignore_index=True)
    monthly_combined = pd.concat([historical_df, forecast_combined], ignore_index=True)
else:
    monthly_combined = historical_df.copy()

print(f"월별 데이터 결합 완료: {len(monthly_combined):,}개 레코드")

# --------------------- 기존 패치(분기/플래그 채우기) ---------------------
monthly_combined['quarter'] = monthly_combined['date'].dt.to_period('Q').astype(str)
monthly_combined['forecast'] = monthly_combined['forecast_flag'].astype(int)
# -----------------------------------------------------------------------

# ======================================================================
# === [PATCH 3] : 업로드 '직전' 파이썬 측 중복 제거(최신 input_date만 유지)
# ======================================================================
monthly_combined['forecast_flag'] = monthly_combined['forecast_flag'].fillna(0).astype(int)
monthly_combined = (monthly_combined
    .sort_values(['hs_code_6d', 'date', 'forecast_flag', 'input_date'])
    .drop_duplicates(subset=['hs_code_6d', 'date', 'forecast_flag'], keep='last'))

# ===================== 분기 집계 =====================
monthly_combined['quarter_period'] = monthly_combined['date'].dt.to_period('Q')
quarterly_grouped = (
    monthly_combined
    .groupby(['hs_code_6d', 'quarter_period', 'input_date'], as_index=False)
    .agg({
        'expDlr': 'sum',
        'forecast_flag': 'max'
    })
)

quarterly_grouped['quarter'] = quarterly_grouped['quarter_period'].astype(str)
quarterly_grouped['date'] = quarterly_grouped['quarter_period'].dt.to_timestamp() + pd.offsets.QuarterEnd(0)
quarterly_grouped = quarterly_grouped[['hs_code_6d', 'quarter', 'expDlr', 'date', 'input_date', 'forecast_flag']]

# === [PATCH 4] : 분기 데이터도 업로드 전 중복 제거
quarterly_grouped['forecast_flag'] = quarterly_grouped['forecast_flag'].fillna(0).astype(int)
quarterly_grouped = (quarterly_grouped
    .sort_values(['hs_code_6d', 'quarter', 'forecast_flag', 'input_date'])
    .drop_duplicates(subset=['hs_code_6d', 'quarter', 'forecast_flag'], keep='last'))

print(f"분기별 집계 완료: {len(quarterly_grouped):,}개 레코드")

# ===================== 월별 테이블 존재/스키마 보강 =====================
print("월별 데이터 테이블 설정 중...")

with engine.connect() as conn:
    monthly_table_exists_query = """
    SELECT COUNT(*) as count
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
    AND table_name = 'us_trade_monthly_data_with_forecast'
    """
    result = conn.execute(text(monthly_table_exists_query))
    monthly_table_exists = result.fetchone()[0] > 0

    if not monthly_table_exists:
        create_monthly_table_query = """
        CREATE TABLE us_trade_monthly_data_with_forecast (
            id INT AUTO_INCREMENT PRIMARY KEY,
            hs_code_6d VARCHAR(10) NOT NULL,
            date DATE NOT NULL,
            expDlr FLOAT NOT NULL,
            input_date DATE NOT NULL,
            forecast_flag TINYINT(1) NOT NULL DEFAULT 0,         -- [PATCH: NOT NULL]
            quarter VARCHAR(10) NOT NULL DEFAULT '1970Q1',
            forecast TINYINT(1) NOT NULL DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE KEY unique_monthly_record (hs_code_6d, date, input_date),
            UNIQUE KEY uniq_monthly_biz (hs_code_6d, date, forecast_flag),
            INDEX idx_monthly_hs_code_date (hs_code_6d, date),
            INDEX idx_monthly_input_date (input_date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        conn.execute(text(create_monthly_table_query))
        print("새 월별 테이블 생성 완료")
    else:
        monthly_column_check_query = """
        SELECT COLUMN_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = 'us_trade_monthly_data_with_forecast'
        """
        result = conn.execute(text(monthly_column_check_query))
        monthly_existing_columns = [row[0] for row in result.fetchall()]

        if 'input_date' not in monthly_existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_monthly_data_with_forecast "
                "ADD COLUMN input_date DATE NOT NULL DEFAULT '2025-01-01'"
            ))
            print("월별 테이블에 input_date 컬럼 추가 완료")

        if 'forecast_flag' not in monthly_existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_monthly_data_with_forecast "
                "ADD COLUMN forecast_flag TINYINT(1) NOT NULL DEFAULT 0"  # [PATCH: NOT NULL]
            ))
            print("월별 테이블에 forecast_flag 컬럼 추가 완료")

        if 'created_at' not in monthly_existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_monthly_data_with_forecast "
                "ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
            ))
            print("월별 테이블에 created_at 컬럼 추가 완료")

        if 'quarter' not in monthly_existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_monthly_data_with_forecast "
                "ADD COLUMN quarter VARCHAR(10) NOT NULL DEFAULT '1970Q1'"
            ))
            print("월별 테이블에 quarter 컬럼 추가 완료")

        if 'forecast' not in monthly_existing_columns:
            conn.execute(text(
                "ALTER TABLE us_trade_monthly_data_with_forecast "
                "ADD COLUMN forecast TINYINT(1) NOT NULL DEFAULT 0"
            ))
            print("월별 테이블에 forecast 컬럼 추가 완료")

        # (기존 코드의 try/except 대신 견고한 유틸 호출)
        ensure_unique_keys_monthly(conn)   # === [PATCH 5] 월별 유니크 보강/정리

    conn.commit()

print("월별 테이블 설정 완료")

# ===================== 월별 데이터 업로드 =====================
print("월별 데이터 업로드 시작...")
monthly_upload_success = False

try:
    temp_monthly_table = f"temp_monthly_forecast_{int(datetime.now().timestamp())}"

    monthly_combined.to_sql(
        name=temp_monthly_table,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'hs_code_6d': sqlalchemy.types.String(length=10),
            'date': sqlalchemy.types.Date(),
            'expDlr': sqlalchemy.types.Float(),
            'input_date': sqlalchemy.types.Date(),
            'forecast_flag': sqlalchemy.types.BOOLEAN(),
            'quarter': sqlalchemy.types.String(length=10),
            'forecast': sqlalchemy.types.BOOLEAN()
        }
    )
    print(f"임시 월별 테이블 {temp_monthly_table} 생성 완료")

    with engine.connect() as conn:
        monthly_upsert_query = f"""
        INSERT INTO us_trade_monthly_data_with_forecast
        (hs_code_6d, date, expDlr, input_date, forecast_flag, quarter, forecast)
        SELECT hs_code_6d, date, expDlr, input_date, forecast_flag, quarter, forecast
        FROM {temp_monthly_table}
        ON DUPLICATE KEY UPDATE
            expDlr = VALUES(expDlr),
            input_date = VALUES(input_date),
            quarter = VALUES(quarter),
            forecast = VALUES(forecast),
            forecast_flag = VALUES(forecast_flag);
        """
        conn.execute(text(monthly_upsert_query))
        conn.execute(text(f"DROP TABLE {temp_monthly_table}"))
        conn.commit()

    print("월별 데이터 업로드 완료!")
    monthly_upload_success = True

except Exception as e:
    print(f"월별 데이터 업로드 오류: {e}")
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_monthly_table}"))
            conn.commit()
        print("임시 월별 테이블 정리 완료")
    except Exception:
        pass

# ===================== 분기 데이터 업로드 =====================
print("데이터베이스 업로드 시작...")
upload_success = False

try:
    temp_table = f"temp_forecast_{int(datetime.now().timestamp())}"

    quarterly_grouped.to_sql(
        name=temp_table,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'hs_code_6d': sqlalchemy.types.String(length=10),
            'quarter': sqlalchemy.types.String(length=10),
            'expDlr': sqlalchemy.types.Float(),
            'date': sqlalchemy.types.Date(),
            'input_date': sqlalchemy.types.Date(),
            'forecast_flag': sqlalchemy.types.BOOLEAN()
        }
    )
    print(f"임시 테이블 {temp_table} 생성 완료")

    with engine.connect() as conn:
        # === [PATCH 6] : 분기 유니크는 이미 ensure 함수로 보장됨 → 재추가 시도 제거
        upsert_query = f"""
        INSERT INTO us_trade_quarter_data_with_forecast
        (hs_code_6d, quarter, expDlr, date, input_date, forecast_flag)
        SELECT hs_code_6d, quarter, expDlr, date, input_date, forecast_flag
        FROM {temp_table}
        ON DUPLICATE KEY UPDATE
            expDlr = VALUES(expDlr),
            date = VALUES(date),
            input_date = VALUES(input_date),
            forecast_flag = VALUES(forecast_flag);
        """
        conn.execute(text(upsert_query))
        conn.execute(text(f"DROP TABLE {temp_table}"))
        conn.commit()

    print("분기 데이터 업로드 완료!")
    upload_success = True

except Exception as e:
    print(f"데이터베이스 업로드 오류: {e}")
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_table}"))
            conn.commit()
        print("임시 테이블 정리 완료")
    except Exception:
        pass

# ===================== 실행 결과 요약 =====================
print("\n" + "="*60)
print("SARIMA 예측 시스템 실행 완료 보고서")
print("="*60)
print(f"실행 일시: {input_date}")
print(f"총 HS Code 수: {len(valid_codes):,}개")
print(f"예측 성공: {model_count:,}개")
print(f"로그 변환 사용: {log_used_count:,}개")
print(f"로그 변환 적용 HS Code: {LOG_TRANSFORM_CODES}")

historical_count = len(quarterly_grouped[quarterly_grouped['forecast_flag'] == 0])
forecast_count = len(quarterly_grouped[quarterly_grouped['forecast_flag'] == 1])
print(f"과거 데이터: {historical_count:,}개 분기별 레코드")
print(f"예측 데이터: {forecast_count:,}개 분기별 레코드")
print(f"총 분기별 레코드: {len(quarterly_grouped):,}개")

monthly_historical_count = len(monthly_combined[monthly_combined['forecast_flag'] == 0])
monthly_forecast_count = len(monthly_combined[monthly_combined['forecast_flag'] == 1])
print(f"월별 과거 데이터: {monthly_historical_count:,}개 레코드")
print(f"월별 예측 데이터: {monthly_forecast_count:,}개 레코드")
print(f"총 월별 레코드: {len(monthly_combined):,}개")
print("="*60)

if upload_success and monthly_upload_success:
    print("\n✅ 분기별 및 월별 데이터가 모두 성공적으로 업로드되었습니다.")
    print("   - 분기별: us_trade_quarter_data_with_forecast")
    print("   - 월별  : us_trade_monthly_data_with_forecast")
elif upload_success:
    print("\n⚠️ 분기별 데이터는 성공, 월별 데이터 업로드 중 오류가 발생했습니다.")
elif monthly_upload_success:
    print("\n⚠️ 월별 데이터는 성공, 분기별 데이터 업로드 중 오류가 발생했습니다.")
else:
    print("\n❌ 분기별 및 월별 데이터 업로드 모두 오류가 발생했습니다.")

print("프로그램 실행 완료")



=== SARIMA 예측 시스템 시작 (2025-09-02) ===
로그 변환 적용 HS Code: ['851762']
데이터베이스 테이블 확인 중...
분기: uniq_quarter_biz 추가 실패(중복/기존 존재). 중복 정리 후 재시도...
분기: uniq_quarter_biz UNIQUE KEY 재시도 성공
분기 테이블 설정 완료
무역 데이터 로딩 중...
✅ 'us_trade_data' 테이블에서 67560건의 데이터를 가져왔습니다.
총 67,560개 레코드 로드 완료
유효한 HS Code (6자리): 478개
총 478개 HS Code에 대해 SARIMA 예측 시작...


SARIMA 예측 진행: 100%|██████████| 478/478 [40:18<00:00,  5.06s/it] 


예측 완료: 447개 HS Code
로그 변환 사용: 1개 HS Code
월별 데이터 결합 완료: 74,265개 레코드
분기별 집계 완료: 24,759개 레코드
월별 데이터 테이블 설정 중...
월별: uniq_monthly_biz 추가 실패(중복/기존 존재). 중복 정리 후 재시도...
월별: uniq_monthly_biz UNIQUE KEY 재시도 성공
월별 테이블 설정 완료
월별 데이터 업로드 시작...
임시 월별 테이블 temp_monthly_forecast_1756812916 생성 완료
월별 데이터 업로드 완료!
데이터베이스 업로드 시작...
임시 테이블 temp_forecast_1756812923 생성 완료
분기 데이터 업로드 완료!

SARIMA 예측 시스템 실행 완료 보고서
실행 일시: 2025-09-02
총 HS Code 수: 478개
예측 성공: 447개
로그 변환 사용: 1개
로그 변환 적용 HS Code: ['851762']
과거 데이터: 22,524개 분기별 레코드
예측 데이터: 2,235개 분기별 레코드
총 분기별 레코드: 24,759개
월별 과거 데이터: 67,560개 레코드
월별 예측 데이터: 6,705개 레코드
총 월별 레코드: 74,265개

✅ 분기별 및 월별 데이터가 모두 성공적으로 업로드되었습니다.
   - 분기별: us_trade_quarter_data_with_forecast
   - 월별  : us_trade_monthly_data_with_forecast
프로그램 실행 완료


In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from DATA.stock_invest_function import *
from statsmodels.stats.diagnostic import acorr_ljungbox
from datetime import datetime
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sqlalchemy import create_engine, text
import sqlalchemy
import matplotlib
import matplotlib.pyplot as plt
import warnings

# 설정
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

# 설정 변수
LOG_TRANSFORM_CODES = ['851762']  # 로그 변환을 적용할 HS Code 리스트
FORECAST_STEPS = 15  # 예측할 개월 수
MIN_PERIODS = 60  # 최소 데이터 개수 (5년)

# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# 실행 날짜
input_date = datetime.now().date()

print(f"=== SARIMA 예측 시스템 시작 ({input_date}) ===")
print(f"로그 변환 적용 HS Code: {LOG_TRANSFORM_CODES}")

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 데이터베이스 테이블 설정 및 컬럼 확인/추가
print("데이터베이스 테이블 확인 중...")

with engine.connect() as conn:
    # 테이블 존재 여부 확인
    table_exists_query = """
    SELECT COUNT(*) as count
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
    AND table_name = 'us_trade_quarter_data_with_forecast'
    """
    result = conn.execute(text(table_exists_query))
    table_exists = result.fetchone()[0] > 0

    if not table_exists:
        # 테이블이 없으면 새로 생성
        create_table_query = """
        CREATE TABLE us_trade_quarter_data_with_forecast (
            id INT AUTO_INCREMENT PRIMARY KEY,
            hs_code_6d VARCHAR(10) NOT NULL,
            quarter VARCHAR(10) NOT NULL,
            expDlr FLOAT NOT NULL,
            date DATE NOT NULL,
            input_date DATE NOT NULL,
            forecast_flag TINYINT(1) DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE KEY unique_record (hs_code_6d, quarter, input_date),
            INDEX idx_hs_code_date (hs_code_6d, date),
            INDEX idx_input_date (input_date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        conn.execute(text(create_table_query))
        print("새 테이블 생성 완료")
    else:
        # 테이블이 있으면 컬럼 확인
        column_check_query = """
        SELECT COLUMN_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = 'us_trade_quarter_data_with_forecast'
        """
        result = conn.execute(text(column_check_query))
        existing_columns = [row[0] for row in result.fetchall()]

        # input_date 컬럼이 없으면 추가
        if 'input_date' not in existing_columns:
            alter_query1 = "ALTER TABLE us_trade_quarter_data_with_forecast ADD COLUMN input_date DATE NOT NULL DEFAULT '2025-01-01'"
            conn.execute(text(alter_query1))
            print("input_date 컬럼 추가 완료")

        # forecast_flag 컬럼이 없으면 추가
        if 'forecast_flag' not in existing_columns:
            alter_query2 = "ALTER TABLE us_trade_quarter_data_with_forecast ADD COLUMN forecast_flag TINYINT(1) DEFAULT 0"
            conn.execute(text(alter_query2))
            print("forecast_flag 컬럼 추가 완료")

        # created_at 컬럼이 없으면 추가
        if 'created_at' not in existing_columns:
            alter_query3 = "ALTER TABLE us_trade_quarter_data_with_forecast ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
            conn.execute(text(alter_query3))
            print("created_at 컬럼 추가 완료")

        # UNIQUE KEY 확인 및 추가 (오류 무시)
        try:
            unique_key_query = """
            ALTER TABLE us_trade_quarter_data_with_forecast
            ADD UNIQUE KEY unique_record (hs_code_6d, quarter, input_date)
            """
            conn.execute(text(unique_key_query))
            print("UNIQUE KEY 추가 완료")
        except:
            print("UNIQUE KEY는 이미 존재합니다")

    conn.commit()

print("테이블 설정 완료")

# 무역 데이터 로드
print("무역 데이터 로딩 중...")
trade_df = fetch_table_data(db_info, "us_trade_data")
trade_df['hs_code_6d'] = trade_df['hs_code'].astype(str).str[:6]
trade_df['date'] = pd.to_datetime(trade_df['date'])

print(f"총 {len(trade_df):,}개 레코드 로드 완료")
print(f"유효한 HS Code (6자리): {len(trade_df['hs_code_6d'].unique()):,}개")

# 유효한 HS Code 리스트
valid_codes = trade_df['hs_code_6d'].unique().tolist()

# SARIMA 예측 함수 정의
def forecast_sarima(df, date_col='date', value_col='expDlr', steps=14, use_log=False):
    try:
        # 월별 집계
        ts = df.groupby(date_col)[value_col].sum().asfreq('M')

        # 데이터 품질 체크
        if ts.isnull().any() or len(ts.dropna()) < MIN_PERIODS:
            return pd.Series(dtype='float64')

        # 로그 변환
        if use_log:
            ts = np.log(ts)

        # SARIMA 파라미터 그리드
        p = d = q = [0, 1]
        P = D = Q = [0, 1]
        s = 12

        param_combinations = list(product(p, d, q))
        seasonal_combinations = list(product(P, D, Q))
        total_combinations = list(product(param_combinations, seasonal_combinations))

        best_aic = np.inf
        best_model = None

        # 최적 모델 탐색
        for (order, seasonal) in total_combinations:
            seasonal_order = (*seasonal, s)
            try:
                model = SARIMAX(ts, order=order, seasonal_order=seasonal_order)
                result = model.fit(disp=False, maxiter=50)

                if result.aic < best_aic:
                    best_aic = result.aic
                    best_model = result

            except:
                continue

        if best_model is None:
            return pd.Series(dtype='float64')

        # 예측 수행
        forecast = best_model.forecast(steps=steps)

        # 로그 변환 역변환
        if use_log:
            forecast = np.exp(forecast)

        # 예측 날짜 인덱스 생성
        forecast.index = pd.date_range(
            start=ts.index[-1] + pd.offsets.MonthEnd(1),
            periods=steps,
            freq='M'
        )

        return forecast

    except Exception as e:
        return pd.Series(dtype='float64')

# SARIMA 예측 실행
print(f"총 {len(valid_codes):,}개 HS Code에 대해 SARIMA 예측 시작...")

forecast_list = []
model_count = 0
log_used_count = 0

for code in tqdm(valid_codes, desc="SARIMA 예측 진행"):
    # HS Code별 데이터 필터링
    sub_df = trade_df[trade_df['hs_code_6d'] == code].copy()

    # 로그 변환 옵션 확인
    use_log = code in LOG_TRANSFORM_CODES
    if use_log:
        log_used_count += 1

    # 예측 실행
    forecast = forecast_sarima(
        sub_df[['date', 'expDlr']],
        steps=FORECAST_STEPS,
        use_log=use_log
    )

    if not forecast.empty:
        # 예측 결과 저장
        temp_df = pd.DataFrame({
            'hs_code_6d': code,
            'date': forecast.index,
            'expDlr': forecast.values,
            'forecast_flag': 1,
            'input_date': input_date
        })
        forecast_list.append(temp_df)
        model_count += 1

print(f"예측 완료: {model_count}개 HS Code")
print(f"로그 변환 사용: {log_used_count}개 HS Code")

# 기존 과거 데이터 준비
historical_df = trade_df.groupby(['hs_code_6d', 'date'], as_index=False)['expDlr'].sum()
historical_df['forecast_flag'] = 0
historical_df['input_date'] = input_date

# 예측 데이터 결합
if forecast_list:
    forecast_combined = pd.concat(forecast_list, ignore_index=True)
    monthly_combined = pd.concat([historical_df, forecast_combined], ignore_index=True)
else:
    monthly_combined = historical_df.copy()

print(f"월별 데이터 결합 완료: {len(monthly_combined):,}개 레코드")

# 분기별 집계
monthly_combined['quarter'] = monthly_combined['date'].dt.to_period('Q')
quarterly_grouped = (
    monthly_combined
    .groupby(['hs_code_6d', 'quarter', 'input_date'], as_index=False)
    .agg({
        'expDlr': 'sum',
        'forecast_flag': 'max'  # 분기 내 하나라도 예측이면 1
    })
)

# 분기 말 날짜 계산
quarterly_grouped['date'] = quarterly_grouped['quarter'].dt.to_timestamp() + pd.offsets.QuarterEnd(0)

print(f"분기별 집계 완료: {len(quarterly_grouped):,}개 레코드")

# 개선된 데이터베이스 업로드 #################################################################################
# 월별 데이터 테이블 설정 및 업로드
print("월별 데이터 테이블 설정 중...")

with engine.connect() as conn:
    # 월별 데이터 테이블 존재 여부 확인
    monthly_table_exists_query = """
    SELECT COUNT(*) as count
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
    AND table_name = 'us_trade_monthly_data_with_forecast'
    """
    result = conn.execute(text(monthly_table_exists_query))
    monthly_table_exists = result.fetchone()[0] > 0

    if not monthly_table_exists:
        # 월별 테이블이 없으면 새로 생성
        create_monthly_table_query = """
        CREATE TABLE us_trade_monthly_data_with_forecast (
            id INT AUTO_INCREMENT PRIMARY KEY,
            hs_code_6d VARCHAR(10) NOT NULL,
            date DATE NOT NULL,
            expDlr FLOAT NOT NULL,
            input_date DATE NOT NULL,
            forecast_flag TINYINT(1) DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE KEY unique_monthly_record (hs_code_6d, date, input_date),
            INDEX idx_monthly_hs_code_date (hs_code_6d, date),
            INDEX idx_monthly_input_date (input_date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        conn.execute(text(create_monthly_table_query))
        print("새 월별 테이블 생성 완료")
    else:
        # 월별 테이블이 있으면 컬럼 확인
        monthly_column_check_query = """
        SELECT COLUMN_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = 'us_trade_monthly_data_with_forecast'
        """
        result = conn.execute(text(monthly_column_check_query))
        monthly_existing_columns = [row[0] for row in result.fetchall()]

        # input_date 컬럼이 없으면 추가
        if 'input_date' not in monthly_existing_columns:
            alter_monthly_query1 = "ALTER TABLE us_trade_monthly_data_with_forecast ADD COLUMN input_date DATE NOT NULL DEFAULT '2025-01-01'"
            conn.execute(text(alter_monthly_query1))
            print("월별 테이블에 input_date 컬럼 추가 완료")

        # forecast_flag 컬럼이 없으면 추가
        if 'forecast_flag' not in monthly_existing_columns:
            alter_monthly_query2 = "ALTER TABLE us_trade_monthly_data_with_forecast ADD COLUMN forecast_flag TINYINT(1) DEFAULT 0"
            conn.execute(text(alter_monthly_query2))
            print("월별 테이블에 forecast_flag 컬럼 추가 완료")

        # created_at 컬럼이 없으면 추가
        if 'created_at' not in monthly_existing_columns:
            alter_monthly_query3 = "ALTER TABLE us_trade_monthly_data_with_forecast ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
            conn.execute(text(alter_monthly_query3))
            print("월별 테이블에 created_at 컬럼 추가 완료")

        # UNIQUE KEY 확인 및 추가 (오류 무시)
        try:
            monthly_unique_key_query = """
            ALTER TABLE us_trade_monthly_data_with_forecast
            ADD UNIQUE KEY unique_monthly_record (hs_code_6d, date, input_date)
            """
            conn.execute(text(monthly_unique_key_query))
            print("월별 테이블에 UNIQUE KEY 추가 완료")
        except:
            print("월별 테이블의 UNIQUE KEY는 이미 존재합니다")

    conn.commit()

print("월별 테이블 설정 완료")

# 월별 데이터 업로드
print("월별 데이터 업로드 시작...")

try:
    # 임시 월별 테이블에 먼저 업로드
    temp_monthly_table = f"temp_monthly_forecast_{int(datetime.now().timestamp())}"

    monthly_combined.to_sql(
        name=temp_monthly_table,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'hs_code_6d': sqlalchemy.types.String(length=10),
            'date': sqlalchemy.types.Date(),
            'expDlr': sqlalchemy.types.Float(),
            'input_date': sqlalchemy.types.Date(),
            'forecast_flag': sqlalchemy.types.BOOLEAN()
        }
    )
    print(f"임시 월별 테이블 {temp_monthly_table} 생성 완료")

    # 기존 월별 데이터 확인 후 UPSERT 또는 INSERT 결정
    with engine.connect() as conn:
        # 기존 월별 테이블에 데이터가 있는지 확인
        monthly_count_query = "SELECT COUNT(*) as count FROM us_trade_monthly_data_with_forecast"
        result = conn.execute(text(monthly_count_query))
        existing_monthly_count = result.fetchone()[0]

        print(f"기존 월별 테이블 레코드 수: {existing_monthly_count:,}개")

        if existing_monthly_count > 0:
            # 기존 데이터가 있으면 UPSERT
            monthly_upsert_query = f"""
            INSERT INTO us_trade_monthly_data_with_forecast
            (hs_code_6d, date, expDlr, input_date, forecast_flag)
            SELECT hs_code_6d, date, expDlr, input_date, forecast_flag
            FROM {temp_monthly_table}
            ON DUPLICATE KEY UPDATE
                expDlr = VALUES(expDlr),
                forecast_flag = VALUES(forecast_flag);
            """
            print("월별 데이터 UPSERT 방식으로 업로드 중...")
        else:
            # 기존 데이터가 없으면 단순 INSERT
            monthly_upsert_query = f"""
            INSERT INTO us_trade_monthly_data_with_forecast
            (hs_code_6d, date, expDlr, input_date, forecast_flag)
            SELECT hs_code_6d, date, expDlr, input_date, forecast_flag
            FROM {temp_monthly_table};
            """
            print("월별 데이터 INSERT 방식으로 업로드 중...")

        monthly_result = conn.execute(text(monthly_upsert_query))
        monthly_affected_rows = monthly_result.rowcount

        # 최종 월별 레코드 수 확인
        final_monthly_count_result = conn.execute(text("SELECT COUNT(*) as count FROM us_trade_monthly_data_with_forecast"))
        final_monthly_count = final_monthly_count_result.fetchone()[0]

        # 임시 월별 테이블 삭제
        conn.execute(text(f"DROP TABLE {temp_monthly_table}"))
        conn.commit()

    print(f"월별 데이터 업로드 완료!")
    print(f"처리된 월별 레코드: {monthly_affected_rows:,}개")
    print(f"월별 테이블 총 레코드: {final_monthly_count:,}개")
    monthly_upload_success = True

except Exception as e:
    print(f"월별 데이터 업로드 오류: {e}")

    # 오류 발생 시 임시 월별 테이블 정리
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_monthly_table}"))
            conn.commit()
        print("임시 월별 테이블 정리 완료")
    except:
        pass

    monthly_upload_success = False

# 개선된 데이터베이스 업로드
print("데이터베이스 업로드 시작...")

try:
    # 임시 테이블에 먼저 업로드
    temp_table = f"temp_forecast_{int(datetime.now().timestamp())}"

    quarterly_grouped.to_sql(
        name=temp_table,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'hs_code_6d': sqlalchemy.types.String(length=10),
            'quarter': sqlalchemy.types.String(length=10),
            'expDlr': sqlalchemy.types.Float(),
            'date': sqlalchemy.types.Date(),
            'input_date': sqlalchemy.types.Date(),
            'forecast_flag': sqlalchemy.types.BOOLEAN()
        }
    )
    print(f"임시 테이블 {temp_table} 생성 완료")

    # 기존 데이터 확인 후 UPSERT 또는 INSERT 결정
    with engine.connect() as conn:
        # 기존 테이블에 데이터가 있는지 확인
        count_query = "SELECT COUNT(*) as count FROM us_trade_quarter_data_with_forecast"
        result = conn.execute(text(count_query))
        existing_count = result.fetchone()[0]

        print(f"기존 테이블 레코드 수: {existing_count:,}개")

        if existing_count > 0:
            # 기존 데이터가 있으면 UPSERT (created_at 업데이트 제거)
            upsert_query = f"""
            INSERT INTO us_trade_quarter_data_with_forecast
            (hs_code_6d, quarter, expDlr, date, input_date, forecast_flag)
            SELECT hs_code_6d, quarter, expDlr, date, input_date, forecast_flag
            FROM {temp_table}
            ON DUPLICATE KEY UPDATE
                expDlr = VALUES(expDlr),
                date = VALUES(date),
                forecast_flag = VALUES(forecast_flag);
            """
            print("UPSERT 방식으로 데이터 업로드 중...")
        else:
            # 기존 데이터가 없으면 단순 INSERT
            upsert_query = f"""
            INSERT INTO us_trade_quarter_data_with_forecast
            (hs_code_6d, quarter, expDlr, date, input_date, forecast_flag)
            SELECT hs_code_6d, quarter, expDlr, date, input_date, forecast_flag
            FROM {temp_table};
            """
            print("INSERT 방식으로 데이터 업로드 중...")

        result = conn.execute(text(upsert_query))
        affected_rows = result.rowcount

        # 최종 레코드 수 확인
        final_count_result = conn.execute(text("SELECT COUNT(*) as count FROM us_trade_quarter_data_with_forecast"))
        final_count = final_count_result.fetchone()[0]

        # 임시 테이블 삭제
        conn.execute(text(f"DROP TABLE {temp_table}"))
        conn.commit()

    print(f"데이터베이스 업로드 완료!")
    print(f"처리된 레코드: {affected_rows:,}개")
    print(f"테이블 총 레코드: {final_count:,}개")
    upload_success = True

except Exception as e:
    print(f"데이터베이스 업로드 오류: {e}")

    # 오류 발생 시 임시 테이블 정리
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_table}"))
            conn.commit()
        print("임시 테이블 정리 완료")
    except:
        pass

    upload_success = False

# 실행 결과 요약
print("\n" + "="*60)
print("SARIMA 예측 시스템 실행 완료 보고서")
print("="*60)
print(f"실행 일시: {input_date}")
print(f"총 HS Code 수: {len(valid_codes):,}개")
print(f"예측 성공: {model_count:,}개")
print(f"로그 변환 사용: {log_used_count:,}개")
print(f"로그 변환 적용 HS Code: {LOG_TRANSFORM_CODES}")

# 데이터 현황
historical_count = len(quarterly_grouped[quarterly_grouped['forecast_flag'] == 0])
forecast_count = len(quarterly_grouped[quarterly_grouped['forecast_flag'] == 1])

print(f"과거 데이터: {historical_count:,}개 분기별 레코드")
print(f"예측 데이터: {forecast_count:,}개 분기별 레코드")
print(f"총 분기별 레코드: {len(quarterly_grouped):,}개")

# 월별 데이터 현황
monthly_historical_count = len(monthly_combined[monthly_combined['forecast_flag'] == 0])
monthly_forecast_count = len(monthly_combined[monthly_combined['forecast_flag'] == 1])
print(f"월별 과거 데이터: {monthly_historical_count:,}개 레코드")
print(f"월별 예측 데이터: {monthly_forecast_count:,}개 레코드")
print(f"총 월별 레코드: {len(monthly_combined):,}개")
print("="*60)

# 업로드 결과 종합
if upload_success and monthly_upload_success:
    print("\n✅ 분기별 및 월별 데이터가 모두 성공적으로 업로드되었습니다.")
    print("   - 분기별 데이터: us_trade_quarter_data_with_forecast")
    print("   - 월별 데이터: us_trade_monthly_data_with_forecast")
elif upload_success:
    print("\n⚠️ 분기별 데이터는 성공, 월별 데이터 업로드 중 오류가 발생했습니다.")
elif monthly_upload_success:
    print("\n⚠️ 월별 데이터는 성공, 분기별 데이터 업로드 중 오류가 발생했습니다.")
else:
    print("\n❌ 분기별 및 월별 데이터 업로드 모두 오류가 발생했습니다.")

print("프로그램 실행 완료")

In [32]:
# 월별 데이터 테이블 설정 및 업로드
print("월별 데이터 테이블 설정 중...")

with engine.connect() as conn:
    # 월별 데이터 테이블 존재 여부 확인
    monthly_table_exists_query = """
    SELECT COUNT(*) as count
    FROM information_schema.tables
    WHERE table_schema = DATABASE()
    AND table_name = 'us_trade_monthly_data_with_forecast'
    """
    result = conn.execute(text(monthly_table_exists_query))
    monthly_table_exists = result.fetchone()[0] > 0

    if not monthly_table_exists:
        # 월별 테이블이 없으면 새로 생성
        create_monthly_table_query = """
        CREATE TABLE us_trade_monthly_data_with_forecast (
            id INT AUTO_INCREMENT PRIMARY KEY,
            hs_code_6d VARCHAR(10) NOT NULL,
            date DATE NOT NULL,
            expDlr FLOAT NOT NULL,
            input_date DATE NOT NULL,
            forecast_flag TINYINT(1) DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE KEY unique_monthly_record (hs_code_6d, date, input_date),
            INDEX idx_monthly_hs_code_date (hs_code_6d, date),
            INDEX idx_monthly_input_date (input_date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        conn.execute(text(create_monthly_table_query))
        print("새 월별 테이블 생성 완료")
    else:
        # 월별 테이블이 있으면 컬럼 확인
        monthly_column_check_query = """
        SELECT COLUMN_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = 'us_trade_monthly_data_with_forecast'
        """
        result = conn.execute(text(monthly_column_check_query))
        monthly_existing_columns = [row[0] for row in result.fetchall()]

        # input_date 컬럼이 없으면 추가
        if 'input_date' not in monthly_existing_columns:
            alter_monthly_query1 = "ALTER TABLE us_trade_monthly_data_with_forecast ADD COLUMN input_date DATE NOT NULL DEFAULT '2025-01-01'"
            conn.execute(text(alter_monthly_query1))
            print("월별 테이블에 input_date 컬럼 추가 완료")

        # forecast_flag 컬럼이 없으면 추가
        if 'forecast_flag' not in monthly_existing_columns:
            alter_monthly_query2 = "ALTER TABLE us_trade_monthly_data_with_forecast ADD COLUMN forecast_flag TINYINT(1) DEFAULT 0"
            conn.execute(text(alter_monthly_query2))
            print("월별 테이블에 forecast_flag 컬럼 추가 완료")

        # created_at 컬럼이 없으면 추가
        if 'created_at' not in monthly_existing_columns:
            alter_monthly_query3 = "ALTER TABLE us_trade_monthly_data_with_forecast ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
            conn.execute(text(alter_monthly_query3))
            print("월별 테이블에 created_at 컬럼 추가 완료")

        # UNIQUE KEY 확인 및 추가 (오류 무시)
        try:
            monthly_unique_key_query = """
            ALTER TABLE us_trade_monthly_data_with_forecast
            ADD UNIQUE KEY unique_monthly_record (hs_code_6d, date, input_date)
            """
            conn.execute(text(monthly_unique_key_query))
            print("월별 테이블에 UNIQUE KEY 추가 완료")
        except:
            print("월별 테이블의 UNIQUE KEY는 이미 존재합니다")

    conn.commit()

print("월별 테이블 설정 완료")

# 월별 데이터 업로드
print("월별 데이터 업로드 시작...")

try:
    # 임시 월별 테이블에 먼저 업로드
    temp_monthly_table = f"temp_monthly_forecast_{int(datetime.now().timestamp())}"

    monthly_combined.to_sql(
        name=temp_monthly_table,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'hs_code_6d': sqlalchemy.types.String(length=10),
            'date': sqlalchemy.types.Date(),
            'expDlr': sqlalchemy.types.Float(),
            'input_date': sqlalchemy.types.Date(),
            'forecast_flag': sqlalchemy.types.BOOLEAN()
        }
    )
    print(f"임시 월별 테이블 {temp_monthly_table} 생성 완료")

    # 기존 월별 데이터 확인 후 UPSERT 또는 INSERT 결정
    with engine.connect() as conn:
        # 기존 월별 테이블에 데이터가 있는지 확인
        monthly_count_query = "SELECT COUNT(*) as count FROM us_trade_monthly_data_with_forecast"
        result = conn.execute(text(monthly_count_query))
        existing_monthly_count = result.fetchone()[0]

        print(f"기존 월별 테이블 레코드 수: {existing_monthly_count:,}개")

        if existing_monthly_count > 0:
            # 기존 데이터가 있으면 UPSERT
            monthly_upsert_query = f"""
            INSERT INTO us_trade_monthly_data_with_forecast
            (hs_code_6d, date, expDlr, input_date, forecast_flag)
            SELECT hs_code_6d, date, expDlr, input_date, forecast_flag
            FROM {temp_monthly_table}
            ON DUPLICATE KEY UPDATE
                expDlr = VALUES(expDlr),
                forecast_flag = VALUES(forecast_flag);
            """
            print("월별 데이터 UPSERT 방식으로 업로드 중...")
        else:
            # 기존 데이터가 없으면 단순 INSERT
            monthly_upsert_query = f"""
            INSERT INTO us_trade_monthly_data_with_forecast
            (hs_code_6d, date, expDlr, input_date, forecast_flag)
            SELECT hs_code_6d, date, expDlr, input_date, forecast_flag
            FROM {temp_monthly_table};
            """
            print("월별 데이터 INSERT 방식으로 업로드 중...")

        monthly_result = conn.execute(text(monthly_upsert_query))
        monthly_affected_rows = monthly_result.rowcount

        # 최종 월별 레코드 수 확인
        final_monthly_count_result = conn.execute(text("SELECT COUNT(*) as count FROM us_trade_monthly_data_with_forecast"))
        final_monthly_count = final_monthly_count_result.fetchone()[0]

        # 임시 월별 테이블 삭제
        conn.execute(text(f"DROP TABLE {temp_monthly_table}"))
        conn.commit()

    print(f"월별 데이터 업로드 완료!")
    print(f"처리된 월별 레코드: {monthly_affected_rows:,}개")
    print(f"월별 테이블 총 레코드: {final_monthly_count:,}개")
    monthly_upload_success = True

except Exception as e:
    print(f"월별 데이터 업로드 오류: {e}")

    # 오류 발생 시 임시 월별 테이블 정리
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_monthly_table}"))
            conn.commit()
        print("임시 월별 테이블 정리 완료")
    except:
        pass

    monthly_upload_success = False

# 개선된 데이터베이스 업로드
print("데이터베이스 업로드 시작...")

try:
    # 임시 테이블에 먼저 업로드
    temp_table = f"temp_forecast_{int(datetime.now().timestamp())}"

    quarterly_grouped.to_sql(
        name=temp_table,
        con=engine,
        if_exists='replace',
        index=False,
        dtype={
            'hs_code_6d': sqlalchemy.types.String(length=10),
            'quarter': sqlalchemy.types.String(length=10),
            'expDlr': sqlalchemy.types.Float(),
            'date': sqlalchemy.types.Date(),
            'input_date': sqlalchemy.types.Date(),
            'forecast_flag': sqlalchemy.types.BOOLEAN()
        }
    )
    print(f"임시 테이블 {temp_table} 생성 완료")

    # 기존 데이터 확인 후 UPSERT 또는 INSERT 결정
    with engine.connect() as conn:
        # 기존 테이블에 데이터가 있는지 확인
        count_query = "SELECT COUNT(*) as count FROM us_trade_quarter_data_with_forecast"
        result = conn.execute(text(count_query))
        existing_count = result.fetchone()[0]

        print(f"기존 테이블 레코드 수: {existing_count:,}개")

        if existing_count > 0:
            # 기존 데이터가 있으면 UPSERT (created_at 업데이트 제거)
            upsert_query = f"""
            INSERT INTO us_trade_quarter_data_with_forecast
            (hs_code_6d, quarter, expDlr, date, input_date, forecast_flag)
            SELECT hs_code_6d, quarter, expDlr, date, input_date, forecast_flag
            FROM {temp_table}
            ON DUPLICATE KEY UPDATE
                expDlr = VALUES(expDlr),
                date = VALUES(date),
                forecast_flag = VALUES(forecast_flag);
            """
            print("UPSERT 방식으로 데이터 업로드 중...")
        else:
            # 기존 데이터가 없으면 단순 INSERT
            upsert_query = f"""
            INSERT INTO us_trade_quarter_data_with_forecast
            (hs_code_6d, quarter, expDlr, date, input_date, forecast_flag)
            SELECT hs_code_6d, quarter, expDlr, date, input_date, forecast_flag
            FROM {temp_table};
            """
            print("INSERT 방식으로 데이터 업로드 중...")

        result = conn.execute(text(upsert_query))
        affected_rows = result.rowcount

        # 최종 레코드 수 확인
        final_count_result = conn.execute(text("SELECT COUNT(*) as count FROM us_trade_quarter_data_with_forecast"))
        final_count = final_count_result.fetchone()[0]

        # 임시 테이블 삭제
        conn.execute(text(f"DROP TABLE {temp_table}"))
        conn.commit()

    print(f"데이터베이스 업로드 완료!")
    print(f"처리된 레코드: {affected_rows:,}개")
    print(f"테이블 총 레코드: {final_count:,}개")
    upload_success = True

except Exception as e:
    print(f"데이터베이스 업로드 오류: {e}")

    # 오류 발생 시 임시 테이블 정리
    try:
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {temp_table}"))
            conn.commit()
        print("임시 테이블 정리 완료")
    except:
        pass

    upload_success = False

# 실행 결과 요약
print("\n" + "="*60)
print("SARIMA 예측 시스템 실행 완료 보고서")
print("="*60)
print(f"실행 일시: {input_date}")
print(f"총 HS Code 수: {len(valid_codes):,}개")
print(f"예측 성공: {model_count:,}개")
print(f"로그 변환 사용: {log_used_count:,}개")
print(f"로그 변환 적용 HS Code: {LOG_TRANSFORM_CODES}")

# 데이터 현황
historical_count = len(quarterly_grouped[quarterly_grouped['forecast_flag'] == 0])
forecast_count = len(quarterly_grouped[quarterly_grouped['forecast_flag'] == 1])

print(f"과거 데이터: {historical_count:,}개 분기별 레코드")
print(f"예측 데이터: {forecast_count:,}개 분기별 레코드")
print(f"총 분기별 레코드: {len(quarterly_grouped):,}개")

# 월별 데이터 현황
monthly_historical_count = len(monthly_combined[monthly_combined['forecast_flag'] == 0])
monthly_forecast_count = len(monthly_combined[monthly_combined['forecast_flag'] == 1])
print(f"월별 과거 데이터: {monthly_historical_count:,}개 레코드")
print(f"월별 예측 데이터: {monthly_forecast_count:,}개 레코드")
print(f"총 월별 레코드: {len(monthly_combined):,}개")
print("="*60)

# 업로드 결과 종합
if upload_success and monthly_upload_success:
    print("\n✅ 분기별 및 월별 데이터가 모두 성공적으로 업로드되었습니다.")
    print("   - 분기별 데이터: us_trade_quarter_data_with_forecast")
    print("   - 월별 데이터: us_trade_monthly_data_with_forecast")
elif upload_success:
    print("\n⚠️ 분기별 데이터는 성공, 월별 데이터 업로드 중 오류가 발생했습니다.")
elif monthly_upload_success:
    print("\n⚠️ 월별 데이터는 성공, 분기별 데이터 업로드 중 오류가 발생했습니다.")
else:
    print("\n❌ 분기별 및 월별 데이터 업로드 모두 오류가 발생했습니다.")

print("프로그램 실행 완료")

월별 데이터 테이블 설정 중...
월별 테이블에 input_date 컬럼 추가 완료
월별 테이블에 forecast_flag 컬럼 추가 완료
월별 테이블에 created_at 컬럼 추가 완료
월별 테이블에 UNIQUE KEY 추가 완료
월별 테이블 설정 완료
월별 데이터 업로드 시작...
임시 월별 테이블 temp_monthly_forecast_1756800888 생성 완료
기존 월별 테이블 레코드 수: 67,574개
월별 데이터 UPSERT 방식으로 업로드 중...
월별 데이터 업로드 완료!
처리된 월별 레코드: 74,265개
월별 테이블 총 레코드: 141,839개
데이터베이스 업로드 시작...
임시 테이블 temp_forecast_1756800895 생성 완료
기존 테이블 레코드 수: 47,288개
UPSERT 방식으로 데이터 업로드 중...
데이터베이스 업로드 완료!
처리된 레코드: 24,759개
테이블 총 레코드: 47,288개

SARIMA 예측 시스템 실행 완료 보고서
실행 일시: 2025-09-02
총 HS Code 수: 478개
예측 성공: 447개
로그 변환 사용: 1개
로그 변환 적용 HS Code: ['851762']
과거 데이터: 22,524개 분기별 레코드
예측 데이터: 2,235개 분기별 레코드
총 분기별 레코드: 24,759개
월별 과거 데이터: 67,560개 레코드
월별 예측 데이터: 6,705개 레코드
총 월별 레코드: 74,265개

✅ 분기별 및 월별 데이터가 모두 성공적으로 업로드되었습니다.
   - 분기별 데이터: us_trade_quarter_data_with_forecast
   - 월별 데이터: us_trade_monthly_data_with_forecast
프로그램 실행 완료


In [5]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    # 'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'host' : get_db_host(),
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [6]:
trade_df = fetch_table_data(db_info, "us_trade_data")

valid_codes = trade_df['hs_code'].unique().tolist()

✅ 'us_trade_data' 테이블에서 67560건의 데이터를 가져왔습니다.


In [10]:
def forecast_monthly_sarima_final(df, date_col='date', value_col='expDlr', steps=12, use_log=False):
    ts = df.groupby(date_col)[value_col].sum().asfreq('M')  # 월말 기준 빈도 지정

    if ts.isnull().any() or len(ts.dropna()) < 60:  # 최소 5년(60개월)
        return pd.Series(dtype='float64')

    ts_transformed = np.log(ts) if use_log else ts

    p = d = q = P = D = Q = [0, 1]
    s = 12
    param_combinations = list(product(p, d, q))
    seasonal_combinations = list(product(P, D, Q))
    total_combinations = list(product(param_combinations, seasonal_combinations))

    best_aic = np.inf
    best_model = None
    best_order = None
    best_seasonal = None

    for (order, seasonal) in total_combinations:
        seasonal_order = (*seasonal, s)
        try:
            model = SARIMAX(ts_transformed, order=order, seasonal_order=seasonal_order)
            result = model.fit(disp=False)
            if result.aic < best_aic:
                best_aic = result.aic
                best_order = order
                best_seasonal = seasonal_order
                best_model = result
        except:
            continue

    if best_model is None:
        return pd.Series(dtype='float64')

    forecast_log = best_model.forecast(steps=steps)
    forecast = np.exp(forecast_log) if use_log else forecast_log
    forecast.index = pd.date_range(start=ts.index[-1] + pd.offsets.MonthEnd(1), periods=steps, freq='M')
    return forecast

In [18]:
# SARIMA 예측 실행 (5년 이상, 로그 변환 X, 경고 제거 O)
forecast_month_list = []

for code in tqdm(valid_codes, desc="SARIMA 예측 중..."):
    trade_df['hs_code_6d'] = trade_df['hs_code'].astype(str).str[:6]
    sub_df = trade_df[trade_df['hs_code_6d'] == code].copy()
    forecast = forecast_monthly_sarima_final(sub_df[['date', 'expDlr']], steps=14, use_log=False)
    if not forecast.empty:
        temp = pd.DataFrame({
            'hs_code_6d': code,
            'date': forecast.index,
            'expDlr': forecast.values,
            'forecast': 1
        })
        forecast_month_list.append(temp)

# 기존 월별 데이터에 forecast=0
historical_df = trade_df[['hs_code_6d', 'date', 'expDlr']].copy()
historical_df['forecast'] = 0

# 전체 월별 데이터 결합
monthly_combined = pd.concat([historical_df] + forecast_month_list, ignore_index=True)
monthly_combined.sort_values(['hs_code_6d', 'date'], inplace=True)

# 분기별 집계
monthly_combined['quarter'] = monthly_combined['date'].dt.to_period('Q')
quarterly_grouped = (
    monthly_combined
    .groupby(['hs_code_6d', 'quarter'], as_index=False)['expDlr']
    .sum()
)

# 분기 말 날짜 계산
quarterly_grouped['date'] = quarterly_grouped['quarter'].dt.to_timestamp() + pd.offsets.QuarterEnd(0)

SARIMA 예측 중...: 100%|██████████| 1/1 [00:18<00:00, 18.93s/it]


In [23]:
# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# DataFrame 저장
quarterly_grouped.to_sql(
    name='us_trade_quarter_data_with_forecast',
    con=engine,
    if_exists='append',  # 'append'로 하면 기존 데이터에 추가
    index=False,
    dtype={
        'hs_code_6d': sqlalchemy.types.String(length=10),
        'quarter': sqlalchemy.types.String(length=10),
        'expDlr': sqlalchemy.types.Float(),
        'date': sqlalchemy.types.Date()
    }
)

print("✅ 데이터가 성공적으로 업로드되었습니다.")

✅ 데이터가 성공적으로 업로드되었습니다.


In [ ]:
forecast_trade_df = fetch_table_data(db_info, "us_trade_monthly_data_with_forecast")

In [25]:
# # ✅ SQLAlchemy 엔진 생성
# engine = create_engine(
#     f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
# )
#
# # ✅ DataFrame 이름 예시 (사용자 변수명 사용)
# # df = your_dataframe  # 예: us_monthly_df
# monthly_combined['date'] = pd.to_datetime(monthly_combined['date'])  # 날짜 타입 보장
#
# # ✅ DB에 저장
# monthly_combined.to_sql(
#     name='us_trade_monthly_data_with_forecast',
#     con=engine,
#     if_exists='replace',  # 기존 테이블 덮어쓰기 (append로 바꾸면 누적 저장 가능)
#     index=False,
#     dtype={
#         'hs_code_6d': sqlalchemy.types.String(length=10),
#         'date': sqlalchemy.types.Date(),
#         'expDlr': sqlalchemy.types.Float(),
#         'forecast': sqlalchemy.types.Integer(),
#         'quarter': sqlalchemy.types.String(length=10)
#     }
# )
#
# print("✅ us_trade_monthly_data_with_forecast 테이블에 데이터 저장 완료.")

✅ us_trade_monthly_data_with_forecast 테이블에 데이터 저장 완료.


In [26]:
forecast_trade_df = fetch_table_data(db_info, "us_trade_monthly_data_with_forecast")

✅ 'us_trade_monthly_data_with_forecast' 테이블에서 67574건의 데이터를 가져왔습니다.


In [27]:
forecast_trade_df[forecast_trade_df['hs_code_6d'] == '851762'].tail(12)

,hs_code_6d,date,expDlr,forecast,quarter
47051,851762,2025-09-30,2.062910e+09,1,2025Q3
47052,851762,2025-10-31,2.246780e+09,1,2025Q4
47053,851762,2025-11-30,2.100540e+09,1,2025Q4
47054,851762,2025-12-31,2.207550e+09,1,2025Q4
47055,851762,2026-01-31,2.260790e+09,1,2026Q1
47056,851762,2026-02-28,2.110310e+09,1,2026Q1
47057,851762,2026-03-31,2.455920e+09,1,2026Q1
47058,851762,2026-04-30,2.260900e+09,1,2026Q2
47059,851762,2026-05-31,2.206470e+09,1,2026Q2
47060,851762,2026-06-30,2.299530e+09,1,2026Q2
